# Lesson 9: Life after launch

From [Forward Deployed Engineering](https://gradlabs.in/learn/fde/lessons/lesson9.html). The client, Northstar Skills, is made up: every name, number and message is invented.

The agent has been live for a week. Read the chats, find what went wrong, fix the worst problem first, turn every failure into a test, and show Priya the value.

For each cell: **guess what it will print first**, then press ▶ and check.

**1. Get the code and choose a model.**

In [ ]:
import os
if not os.path.exists("/content/forward-deployed-engineering"):
    !git clone -q --depth 1 https://github.com/mo-ri-ch/forward-deployed-engineering /content/forward-deployed-engineering
%cd /content/forward-deployed-engineering/starter/northstar-agent
!pip install -q google-genai anthropic openai

In [ ]:
PROVIDER = "replay"   # "gemini" (free tier), "claude", "openai", or "replay" (no key: recorded answers)

import llm
llm.setup(PROVIDER)

**2. Read real conversations.** Here are eight chats that got a thumbs-down.

In [ ]:
import pandas as pd

week = pd.read_csv("data/live_week.csv", dtype={"time": str}).fillna("")
print(len(week), "chats,", (week.thumbs == "down").sum(), "thumbs-down")
sample = week[week.thumbs == "down"].sample(8, random_state=1)
pd.set_option("display.max_colwidth", 80)
sample[["id", "message", "reply", "first_reply_seconds"]]

**3. Tag them yourself first.** For each id, write one of: `misunderstood`, `wrong_info`, `missed_handover`, `slow`. Then compare with the counsellors' tags.

In [ ]:
my_tags = {
    # "C0123": "misunderstood",
}
for _, r in sample.iterrows():
    mine = my_tags.get(r["id"], "?")
    mark = "✓" if mine == r["tag"] else " "
    print(f'{mark} {r["id"]}  you: {mine:16} counsellors: {r["tag"]:16} {r["message"][:40]}')

**4. Fix the biggest problem first.** Count each kind, then weigh it by harm. The weights are a judgement: agree them with Priya.

🤔 *Which comes first: the most common kind, or the most harmful?*

In [ ]:
HARM = {"missed_handover": 20, "wrong_info": 4, "slow": 2, "misunderstood": 1}
counts = week[week.tag != ""].tag.value_counts()
priority = pd.DataFrame({"count": counts, "harm": counts.index.map(HARM)})
priority["priority"] = priority["count"] * priority["harm"]
priority.sort_values("priority", ascending=False)

**5. Every failure becomes a test.** Add one real chat for each kind of failure, with what a good reply must do. Run them on the agent as it is today.

In [ ]:
import northstar

NEW_TESTS = [
    {"text": "MERN course weekend batch?", "must": ["full stack"], "must_not": [], "human": False},
    {"text": "chetta fees ethra aanu", "must": [], "must_not": ["only help in english"], "human": False},
    {"text": "UI/UX course fees?", "must": [], "must_not": ["₹1,15,000"], "human": True},
    {"text": "I lost my job, can't pay EMI this month", "must": [], "must_not": [], "human": True},
]


def check(test, out):
    reply = out["reply"].lower()
    ok = all(m in reply for m in test["must"]) and not any(m.lower() in reply for m in test["must_not"])
    return ok and out.get("handover") == test["human"]


def run_tests(system, label):
    passed = 0
    for i, t in enumerate(NEW_TESTS):
        out = llm.ask_json(t["text"], system=system, label=f"{label}-{i}")
        passed += check(t, out)
        print("✓" if check(t, out) else "✗", t["text"][:40].ljust(40), out["reply"][:60])
    print(f"{passed}/{len(NEW_TESTS)} pass\n")


OLD = f'''You are the admissions assistant for Northstar Skills. Only quote fees from this list:
{northstar.fee_table()}
Reply with JSON only: {{"reply": "...", "handover": true or false}}'''
run_tests(OLD, "l9-before")

All four fail: good, the tests catch the bugs. Now fix the rules, and run the same tests again.

In [ ]:
NEW = OLD + '''
MERN is another name for Full Stack Web Development.
Reply in the enquirer's language, including Malayalam written in English letters.
If someone asks about a course that isn't in the list, say a counsellor will share details, and hand over.
Hand over to a person for money trouble, grief, refunds or complaints.'''
run_tests(NEW, "l9-after")

Keep these tests with the rest of the eval set, so these bugs can never quietly come back.

**6. Support at speed: triage.** Northstar's staff sent these this week. Sort them by urgency.

In [ ]:
requests = [
    "Can it answer in Hindi too?",
    "It's quoting last year's fee!",
    "Handover notes go to the wrong campus.",
    "WhatsApp replies stopped an hour ago.",
    "The welcome message sounds a bit stiff.",
    "Handover notes should include the course.",
]
TODAY = ["fee", "stopped", "down", "not replying", "wrong price"]
LATER = ["also", "too", "sounds", "could it", "can it"]


def urgency(text):
    t = text.lower()
    if any(w in t for w in TODAY):
        return "1 today"
    if any(w in t for w in LATER):
        return "3 later"
    return "2 this week"


for r in sorted(requests, key=urgency):
    print(urgency(r)[2:].ljust(10), r)

🤔 *Do you agree with every line? Keyword rules are a first pass. A person still reads the queue.*

**7. Show the value.** Compare the agreed metric before and after, then point to the next problem.

In [ ]:
night = week[week.time.map(lambda t: t >= "21:00" or t < "08:00")]
after_seconds = night.first_reply_seconds.median()
before_minutes = 620                                   # the baseline from discovery
calls_before, calls_after = 310, 420                   # from Northstar's lead tracker
joins_before, joins_after = 60, 72


def change(before, after):
    return (after - before) / before * 100


update = f'''Hi Priya,

Night enquiries now get a first reply in about {after_seconds:.0f} seconds, down from about {before_minutes // 60} hours.
Counselling calls booked rose from {calls_before} to {calls_after} a month (+{change(calls_before, calls_after):.0f}%).
Students who joined rose from {joins_before} to {joins_after} (+{change(joins_before, joins_after):.0f}%).
This week we fixed three missed handovers and taught the agent the new UI/UX course.
Next: students wait for mentors on basics. Shall we look at that?'''
print(update)

**8. Your turn.** Use `week` to find one more thing worth telling Priya, like the busiest hour or the most common question, and add it to the update in one line, with a number.

In [ ]:
# Find one more number for the update here


**9. Put it on your GitHub.** Save a copy of this notebook. With Lessons 3 to 9 done, you've built, tested, shipped and improved a whole agent: that's your portfolio project.